# Petrophysical Interpretation with Welly

This tutorial demonstrates the petrophysics module in welly, which provides enterprise-grade petrophysical calculations for well log interpretation.

## What you'll learn

- How to set up petrophysical parameters
- Calculate shale volume using multiple methods
- Compute porosity from density, neutron, and sonic logs
- Estimate water saturation using Archie and shaly-sand models
- Calculate permeability from porosity and saturation
- Determine net pay and reservoir quality
- Use the PetroInterpreter for streamlined workflows
- Visualize results with the built-in plot() method

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import welly
from welly import Well
from welly.petro import (
    PetroInterpreter,
    PetrophysicalParameters,
    MatrixParameters,
    FluidParameters,
    ClayParameters,
)

print(f"welly version: {welly.__version__}")

## Load a Well

First, let's load a well from a LAS file.

In [ ]:
well = Well.from_las('data/P-129_out.LAS')
print(f"Well: {well.name}")
print(f"Curves: {list(well.data.keys())}")

## Setting Up Parameters

Petrophysical calculations require various parameters. The `PetrophysicalParameters` class provides a structured way to manage these.

In [ ]:
# Create parameters for a sandstone reservoir
params = PetrophysicalParameters(
    matrix=MatrixParameters.sandstone(),
    fluid=FluidParameters(rw=0.05, rw_temp=75),
    clay=ClayParameters(
        gr_clean=20,
        gr_shale=130,
        nphi_shale=0.35,
        rho_shale=2.55,
        rt_shale=5.0
    ),
    a=0.81,  # Humble formula
    m=2.0,
    n=2.0,
    name='Sandstone Reservoir'
)

print(f"Matrix density: {params.matrix.rho_matrix} g/cc")
print(f"Rw at 75F: {params.fluid.rw} ohm.m")
print(f"Rw at 150F: {params.fluid.rw_at_temperature(150):.4f} ohm.m")

## Using the PetroInterpreter

The `PetroInterpreter` class provides a high-level interface for running petrophysical interpretations.

In [ ]:
# Create interpreter using the well.petro() convenience method
interp = well.petro(params=params)

# Check what inputs are available
status = interp.check_inputs(verbose=True)

## Shale Volume Calculation

Shale volume (Vshale) is typically the first step in petrophysical interpretation.

In [ ]:
# Calculate Vshale using Larionov method
vsh = interp.vshale(method='larionov')

print(f"Vshale computed: {vsh.mnemonic}")
print(f"Range: {np.nanmin(vsh.values):.3f} - {np.nanmax(vsh.values):.3f}")

## Porosity Calculation

In [ ]:
# Calculate density porosity
phi_d = interp.porosity(method='density', output='PHID')

print(f"Density porosity computed: {phi_d.mnemonic}")
print(f"Range: {np.nanmin(phi_d.values):.3f} - {np.nanmax(phi_d.values):.3f}")

## Water Saturation

In [ ]:
# Calculate Sw using Archie
sw = interp.sw(method='archie', phi='PHID')

print(f"Sw computed: {sw.mnemonic}")
print(f"Range: {np.nanmin(sw.values):.3f} - {np.nanmax(sw.values):.3f}")

## Complete Workflow

The `run_standard_interpretation()` method runs a complete interpretation workflow in one call.

In [ ]:
# Load a fresh well
well2 = Well.from_las('data/P-129_out.LAS')
interp2 = well2.petro(params=params)

# Run standard interpretation
results = interp2.run_standard_interpretation(
    vshale_method='larionov',
    porosity_method='density',
    sw_method='archie',
    phi_cutoff=0.08,
    sw_cutoff=0.50,
    vsh_cutoff=0.40
)

print("Curves computed:", results['curves_computed'])
print("Statistics:", results['statistics'])

## Visualizing Results with plot()

The `PetroInterpreter` has a built-in `plot()` method that creates a standard multi-track interpretation plot.

In [ ]:
# Generate the default interpretation plot
fig = interp2.plot()
plt.show()

In [ ]:
# Plot with a specific depth range
fig = interp2.plot(depth_range=(1500, 1800), title='Zoomed Interpretation')
plt.show()

In [ ]:
# Plot only the computed results (no input curves)
fig = interp2.plot(show_inputs=False)
plt.show()

In [ ]:
# Select specific tracks to display
fig = interp2.plot(tracks=['GR', 'VSH', 'PHI', 'SW', 'PAY'])
plt.show()

## Saving Plots

The `plot()` method returns a matplotlib Figure that can be saved to file.

In [ ]:
# Create and save a publication-quality figure
fig = interp2.plot(figsize=(16, 12), title=f'Petrophysical Interpretation: {well2.name}')
fig.savefig('data/interpretation_plot.png', dpi=150, bbox_inches='tight')
print("Saved: data/interpretation_plot.png")
plt.show()

## Summary

The `PetroInterpreter.plot()` method provides:

- Auto-selection of tracks based on available curves
- Proper formatting (log scales for resistivity/permeability, fill patterns)
- Cutoff lines for Vshale, porosity, and Sw
- Customizable depth range, figure size, and track selection
- Returns matplotlib Figure for saving or further customization